# Reading the archive: V6 and V7 data (backup drive or clusters)

Since the data reduction of October 2026, the raw Monte-Carlo files are gone from the clusters. What is left, on Dror's backup drive (folder `Yaniv-Data`) **and** on each cluster under `/home/data/<user>/data`, in the same layout:

- **V7:** the `means_db` (means and error bars of every simulation), plus one `results_slim.jld2` per seed (exact means and the measurement stream averaged into 100 bins);
- **V6:** each campaign's `num_control/sim_info.jld2` (the means), plus the slims.

This notebook reads both, for the same parameters (U = 3t, 14 × 14, n = 0.875, clean, K0-TCA with cluster size 10), and plots the superfluid stiffness of V6 and V7 together. Each cluster can be read from the drive or from the cluster itself (Setup). Everything here only **reads**; the code refuses to write on the drive. Details: [docs/WORKFLOW.md §8](../docs/WORKFLOW.md#8-the-backup-drive).

> **Kernel.** Use a Julia kernel whose environment has JLD2 and Plots, started on that environment, e.g. **"Julia 1.11 default env"**. Not a kernel started with `--project=@.` (the IJulia default): from this folder it picks up the repository's `Project.toml`/`Manifest.toml`, recorded on the cluster with Julia 1.12, and fails to start ("Package Preferences ... is required but does not seem to be installed") unless that environment is instantiated. To make such a kernel, once: `using IJulia; installkernel("Julia 1.11 default env", "--project=@v1.11")`, then pick it in the kernel menu (top right).

## Setup: where to read from

Tell the code where the data is, **before** the `include` (to change it later, restart the kernel). There are no built-in locations. Best: set them once in `~/.julia/config/startup.jl`; otherwise uncomment and fill in the lines below. Each cluster is chosen separately; for each, the first setting that applies wins:

| | Setting | Reads from |
|---|---|---|
| 1 | `LANDAU_ROOT` / `CURIE_ROOT` | any folder holding that cluster's `MC_Hubbard_V7/` and `MC_Hubbard_V6/`: its cluster mount's `data` folder, or a local copy |
| 2 | `BACKUP_DRIVE` | the backup drive's `Yaniv-Data` folder: `<drive>/landau`, `<drive>/curie` |
| 3 | `LANDAU_MOUNT` / `CURIE_MOUNT` | the cluster mounts of `/home/data/<user>`: `<mount>/data` |

The `means_db` queries, the slims and the V6 files all follow this choice.

In [ ]:
# ENV["BACKUP_DRIVE"] = "<where the drive is mounted>/Yaniv-Data"   # the backup drive, both clusters
# ENV["LANDAU_ROOT"]  = "<Landau's mount>/data"                    # Landau from its cluster mount instead
# ENV["CURIE_ROOT"]   = "<Curie's mount>/data"                     # Curie from its cluster mount instead

const REPO = normpath(joinpath(@__DIR__, ".."))   # this notebook lives in <repo>/notebooks
include(joinpath(REPO, "drive_data.jl"))          # also loads means_db.jl (JLD2 only, seconds)

println("reading from:")
foreach(c -> println("  ", rpad(c, 7), isempty(cluster_root(c)) ? "(not set)" : cluster_root(c)), ("landau", "curie"))
db_status()

In [ ]:
using Plots

### What is on the drive (optional; drive only)

`drive_check()` compares every archived raw seed with its slim (~20 s) and lists the few seeds that have no slim. `drive_summary()` gives GB / files / slims per campaign; it stats ~70k files, so it takes a few minutes. Both read the drive (`BACKUP_DRIVE`) only; never run a scan like this over a cluster mount.

In [ ]:
isempty(get(ENV, "BACKUP_DRIVE", "")) || (held = drive_check());
# drive_summary()

## V7: means from `means_db`

A query names the parameters with keywords. **Pin every parameter you are not sweeping**: anything left out is averaged over (the query warns and lists what it mixed).

In [ ]:
sel = (U=3.0, Lx=14, Ly=14, V=0.0, wanted_n=0.875, alpha=1.0, beta_phi=0.0,
       diag_method="K0-TCA", TClustersize=10, mu_disorder="none")

c7 = get_curve(; obs=:stiffness, x=:beta, sel...)   # one point per beta
p7 = sortperm(1 ./ c7.x)
T7, ρ7, e7 = (1 ./ c7.x)[p7], c7.value[p7], c7.error[p7]                                # T, rho_s, error
plot(T7, ρ7, yerror=e7, label="U=3.0, Lx=14, Ly=14, V=0.0", xlabel="T", ylabel="ρ_s", title="Stiffness vs Temperature", legend=:topright)

## V7: the slims of one point

`find_slims` takes the same selectors and returns every seed of the matching simulations, with the path of its slim under that cluster's folder (`exists=false` for the few seeds that have no slim).

In [ ]:
sl = find_slims(; sel..., beta=10.0)

In [ ]:
o = slim_obs(sl[1].slim)            # names only, no data read
println(length(o.means), " observables with exact means, ", length(o.binned), " with binned streams")
println("binned: ", join(o.binned, ", "))

In [ ]:
s = read_slim(sl[1].slim; obs=[:n, :stiffness, :DdD])   # obs=nothing reads everything

println("N = ", s.meta["num_measurements"], " measurements in ", s.meta["num_bins"],
        " bins of ", s.meta["bin_size"], "   (bins decorrelated: ", s.meta["bins_decorrelated"], ")")
println("stiffness = ", s.means["stiffness"]["value"], " ± ", s.means["stiffness"]["error"],
        "   exact, the same as means_db")
println("n: exact mean = ", s.means["n"]["value"], ",  mean of the bins = ", sum(s.binned["n"]) / length(s.binned["n"]))
println("converged mu = ", s.sim_data["mu"])

The stiffness is a nonlinear function of several streams, so it has exact means but no binned stream. Linear observables (`n`, `DdD`, …) can be re-averaged from the bins; error bars should come from `means`.

In [ ]:
histogram(s.binned["DdD"]; marker=:circle, ms=2, xlabel="bin", ylabel="DdD (bin mean)",
     label="$(s.meta["bin_size"]) measurements per bin", title="Binned stream of one seed")

## V6: means from `sim_info.jld2`

V6 keeps one means file per campaign: a Dict `sim_num => Dict(parameters..., "means" => Dict(obs => Dict(value, error, ...)), ...)`. This campaign has 620 simulations; reading it takes about a minute from the drive, longer over a cluster mount.

> The drive has each V6 campaign's *last* `sim_info.jld2`. For the clean-system campaigns (like this one) it is identical to the thesis input; the disorder campaigns were updated on the cluster after the thesis copies were taken (WORKFLOW.md §8.4).

In [ ]:
v6_campaigns(; cluster="curie")

In [ ]:
si = load_v6_sim_info("curie", "Full_Sys_TCA__25-02-2026")   # ~1 min
count(!=("updated_at"), keys(si))                             # number of simulations

In [ ]:
v6 = [sim for (k, sim) in si if k != "updated_at" && haskey(sim, "means") &&
      sim["U"] == sel.U && sim["Lx"] == sel.Lx && sim["V"] == sel.V && sim["wanted_n"] == sel.wanted_n &&
      sim["diag_method"] == sel.diag_method && sim["TClustersize"] == sel.TClustersize]
sort!(v6; by = sim -> 1 / sim["beta"])

T6 = [1 / sim["beta"] for sim in v6]
ρ6 = [sim["means"]["stiffness"]["value"] for sim in v6]
e6 = [sim["means"]["stiffness"]["error"] for sim in v6]
plot(T6, ρ6, yerror=e6, label="U=3.0, Lx=14, Ly=14, V=0.0", xlabel="T", ylabel="ρ_s", title="Stiffness vs Temperature", legend=:topright)

## V6 and V7 together

The same parameters, computed by two versions of the code: the curves should agree within the error bars. The transition is where ρ_s crosses the BKT line 2T/π.

In [ ]:
plt = plot(; xlabel="T / t", ylabel="ρₛ / t", xlims=(0, 0.3), ylims=(0, 0.2), framestyle=:box,
           title="U = 3t, L = 14, n = 0.875")
plot!(plt, T6, ρ6; yerr=e6, marker=:square, ms=3, label="V6 (sim_info)")
plot!(plt, T7, ρ7; yerr=e7, marker=:circle, ms=3, label="V7 (means_db)")
plot!(plt, T -> 2T / π, 0, 0.3; ls=:dash, color=:black, label="2T/π")

## V6: the slims

The V6 slims have the same format as the V7 ones; their means are the thesis-time `means_cache` values (`meta["means_source"] == "means_cache"`).

In [ ]:
vs = v6_seeds("curie", "Full_Sys_TCA__25-02-2026")   # (unit, seed, dir, slim, means_cache)
length(vs)

In [ ]:
s6 = read_slim(vs[1].slim; obs=[:n, :stiffness])
println(s6.meta["means_source"], ":  U = ", s6.sim_data["U"], ", L = ", s6.sim_data["Lx"],
        ", beta = ", s6.sim_data["beta"], ", stiffness = ", s6.means["stiffness"]["value"])

## More

- All `means_db` queries (`get_observable`, `get_spectrum`, `find_all`, disorder averages, …): WORKFLOW.md §7.
- `get_stream` refuses slimmed seeds: the raw streams are gone, so use `read_slim(...).binned`.
- **Over a cluster mount**, keep to small reads (one curve, a few slims, one `sim_info`): every file is opened over SFTP on the cluster's storage master, and bulk reads over a mount have crashed a laptop. For anything big, use the drive or work on the cluster.
- The drive is read-only to the code (`DriveReadOnly`). To change anything, copy the folder to a local disk first.